# 9단계 · 자기지도(MCM 방식) · 준지도(Deep SAD) · 경계선 보정(컨포멀)  [외부 데이터 사용 — B층]

| 파트 | 방법 | 비라벨을 쓰는 방식 | 참고 |
|---|---|---|---|
| A | **변수 가리고 맞히기** (MCM 방식) | 변수 하나를 나머지 23개로 예측하도록 학습 → 예측이 빗나가면 이상 | Yin 외, ICLR 2024 (MCM) |
| B | **Deep SVDD / Deep SAD** | 신경망이 정상 쇼트를 한 점(중심) 근처로 모음. SAD는 라벨 불량을 중심에서 밀어냄 | Ruff 외, ICLR 2020 |
| C | **컨포멀 경계선** | 과거 점수 분포로 "양품의 5%만 경보"가 되도록 경계선을 정함 | Barber 외, Ann. Stat. 2023 |

공통 조건은 7단계와 같습니다. 원래 단위, 정상 모드, **그 쇼트 이전**의 과거 쇼트만 사용합니다.

**준지도 검증 규칙 (중요):** Deep SAD가 쓰는 '라벨 불량'은 **채점하는 날이 아닌 다른 날**의 불량만 씁니다(날짜 블록 단위).
같은 사건의 불량을 학습에 넣으면 3단계처럼 이웃을 외우는 누설이 됩니다.

> 파트 B는 `torch`가 필요합니다. 없으면 `pip install torch` 후 커널을 재시작하세요. 없으면 B만 건너뜁니다.

In [ ]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
from sklearn.linear_model import Ridge
from src.official import *
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

ML = pd.read_csv(RAW / "moldset_labeled.csv"); MLI = ML.set_index("Unnamed: 0")   # [외부]
DATA, BASE = {}, {}
for p in ["CN7", "RG3"]:
    lab = load(p)[0]
    lab["ts"] = pd.to_datetime(lab["order"].map(MLI["TimeStamp"]))
    DATA[p] = lab
    u = pd.read_csv(RAW / f"unlabeled_raw_{p.lower()}.csv"); u["ts"] = pd.to_datetime(u["TimeStamp"])
    BASE[p] = u[u["Barrel_Temperature_1"] > 100].drop_duplicates("ts").sort_values("ts")


class RobustScaler:
    def fit(self, P):
        self.med = np.median(P, axis=0)
        sd = np.maximum(np.median(np.abs(P - self.med), axis=0) * 1.4826, P.std(axis=0))
        self.ok = sd > 1e-6; self.sd = np.where(self.ok, sd, 1)
        return self
    def transform(self, X):
        return np.where(self.ok, np.clip((X - self.med) / self.sd, -20, 20), 0)


def shot_view(lab):
    """쇼트 단위: 원값 X, 시각 T, 쌍 번호, 쇼트 불량 수, 날짜 블록(3시간 넘게 끊기면 새 블록)."""
    sh = lab[~lab["pair"].duplicated()].copy()
    X = MLI.loc[sh["order"], VARS].values
    bad = lab.groupby("pair")["y"].sum().reindex(sh["pair"]).values
    block = (sh["ts"].diff().dt.total_seconds() > 3 * 3600).cumsum().values
    return X, sh["ts"].values, sh["pair"].values, bad, block


def to_rows(lab, pairs, v):
    return lab["pair"].map(pd.Series(v, index=pairs)).values


def report(lab, s, name, p):
    y = lab["y"].values
    e = evaluate(lab, s, name, p, 1)
    o = np.lexsort((np.random.RandomState(0).rand(len(s)), -np.asarray(s, float)))
    k5 = int(round(len(s) * 0.05)); tp = int(y[o[:k5]].sum())
    P = int(y.sum()); ceil5 = 2 * min(P, k5) / (k5 + P)
    sh = pd.Series(s, index=lab.index).groupby(lab["pair"]).first().rank(ascending=False, pct=True) * 100
    both = lab.groupby("pair")["y"].transform("sum") == 2
    return {"모델": name, "제품": p, "AP": e["AP"], "유의": e["유의"], "TP@5%": tp,
            "F1@5%": 2 * tp / (k5 + P), "예산5% 상한 대비": 2 * tp / (k5 + P) / ceil5,
            "둘다불량 순위%": sh[lab.loc[both, "pair"].unique()].round(1).tolist()}
print("준비 완료")

## 파트 A. 변수 가리고 맞히기 (MCM 방식, 자기지도)

쇼트마다 과거 300쇼트로 **24개의 작은 예측 모델**(Ridge 회귀)을 만듭니다. "충전 시간 = 나머지 23개 변수로 예측" 같은 식입니다.
새 쇼트에서 각 변수의 **실제값 − 예측값**(잔차)을 구하고, 잔차가 평소보다 얼마나 큰지 rms로 합칩니다.

7단계 물리 관계는 사람이 고른 관계 5개였고, 이건 **데이터가 스스로 배운 관계 24개**입니다.

> **질문 9-1.** 5단계 강건 z(변수마다 평소 범위)보다 이 방법이 더 잘 잡을 수 있는 불량은 어떤 불량일까요?
> 힌트: 그림 3(충전 시간 vs 사출 속도)에서 둘다불량은 각 변수 범위보다 **직선에서** 크게 벗어났습니다.
>
> 내 예상:

In [ ]:
def masked_scores(lab, base, W=300, refit=10):
    X, T, pairs, bad, block = shot_view(lab)
    bt, BX = base["ts"].values, base[VARS].values
    out = np.zeros(len(X))
    for i in range(len(X)):
        if i % refit == 0:
            P = BX[bt < T[i]][-W:]
            sc = RobustScaler().fit(P); Z = sc.transform(P)
            act = np.where(Z.std(axis=0) > 1e-6)[0]                       # 움직이는 변수만
            regs = {j: Ridge(alpha=1.0).fit(np.delete(Z, j, axis=1), Z[:, j]) for j in act}
            res = np.column_stack([Z[:, j] - regs[j].predict(np.delete(Z, j, axis=1)) for j in act])
            rsc = RobustScaler().fit(res)                                  # 잔차의 평소 크기
        z = sc.transform(X[i])
        r = np.array([z[j] - regs[j].predict(np.delete(z, j)[None])[0] for j in act])
        out[i] = np.sqrt((rsc.transform(r) ** 2).mean())
    return to_rows(lab, pairs, out)

def robust_scores(lab, base, W=300, refit=10):
    """비교용: 5단계와 같은 강건 z (같은 창 W=300)."""
    X, T, pairs, bad, block = shot_view(lab)
    bt, BX = base["ts"].values, base[VARS].values
    out = np.zeros(len(X))
    for i in range(len(X)):
        if i % refit == 0:
            sc = RobustScaler().fit(BX[bt < T[i]][-W:])
        out[i] = np.sqrt((sc.transform(X[i]) ** 2).mean())
    return to_rows(lab, pairs, out)

ROWS, SC = [], {}
for p, lab in DATA.items():
    t0 = time.time()
    SC[(p, "강건 z")] = robust_scores(lab, BASE[p])
    SC[(p, "변수 가리고 맞히기")] = masked_scores(lab, BASE[p])
    for name in ["강건 z", "변수 가리고 맞히기"]:
        ROWS.append(report(lab, SC[(p, name)], name, p))
    print(p, f"{time.time() - t0:.0f}초")
pd.DataFrame(ROWS).round(3)

## 파트 B. Deep SVDD / Deep SAD (준지도)

- **Deep SVDD**: 작은 신경망(24 → 16 → 8)이 과거 정상 쇼트를 8차원 공간의 **중심 한 점 근처로** 모으도록 학습합니다. 중심에서 먼 쇼트 = 이상.
- **Deep SAD**: 위에 더해, **다른 날의 불량 쇼트**를 중심에서 **멀리 밀어내도록** 함께 학습합니다.
- 30쇼트마다 과거 300쇼트로 다시 학습합니다.

> **질문 9-2.** CN7 불량은 전부 10/16 하나의 블록에 있습니다. 10/16 쇼트를 채점할 때 Deep SAD가 쓸 수 있는 '다른 날 불량'은 몇 개일까요?
> 그렇다면 Deep SAD와 Deep SVDD의 결과는 어떻게 될까요?
>
> 내 예상:

In [ ]:
try:
    import torch, torch.nn as nn
    HAS_TORCH = True
except ImportError:
    HAS_TORCH = False
    print("torch 없음 → 파트 B 건너뜀 (pip install torch)")

def train_sad(Xu, Xa, eta=1.0, epochs=40, seed=0):
    """Deep SAD (Ruff 외 2020). Xa 가 비어 있으면 Deep SVDD 와 같다. 반환: 점수 함수."""
    torch.manual_seed(seed)
    net = nn.Sequential(nn.Linear(Xu.shape[1], 16, bias=False), nn.LeakyReLU(), nn.Linear(16, 8, bias=False))
    U = torch.tensor(Xu, dtype=torch.float32)
    A = torch.tensor(Xa, dtype=torch.float32) if len(Xa) else None
    with torch.no_grad():
        c = net(U).mean(0)
        c[(c.abs() < 0.1) & (c < 0)] = -0.1; c[(c.abs() < 0.1) & (c >= 0)] = 0.1   # 중심이 0에 붙지 않게
    opt = torch.optim.Adam(net.parameters(), lr=1e-3, weight_decay=1e-5)
    for _ in range(epochs):
        perm = torch.randperm(len(U))
        for b in range(0, len(U), 128):
            loss = ((net(U[perm[b:b + 128]]) - c) ** 2).sum(1).mean()                 # 정상은 중심으로
            if A is not None:
                loss = loss + eta * (1 / (((net(A) - c) ** 2).sum(1) + 1e-6)).mean()   # 불량은 멀리
            opt.zero_grad(); loss.backward(); opt.step()
    return lambda X: ((net(torch.tensor(X, dtype=torch.float32)) - c) ** 2).sum(1).detach().numpy()

def sad_scores(lab, base, W=300, refit=30, seed=0):
    X, T, pairs, bad, block = shot_view(lab)
    bt, BX = base["ts"].values, base[VARS].values
    s_svdd, s_sad, n_lab = np.zeros(len(X)), np.zeros(len(X)), np.zeros(len(X), int)
    for i in range(len(X)):
        if i % refit == 0:
            P = BX[bt < T[i]][-W:]; sc = RobustScaler().fit(P)
            A = X[(block != block[i]) & (bad > 0)]          # 다른 날 블록의 불량 쇼트만
            f_svdd = train_sad(sc.transform(P), np.zeros((0, len(VARS))), seed=seed)
            f_sad = train_sad(sc.transform(P), sc.transform(A), seed=seed) if len(A) else f_svdd
        z = sc.transform(X[i:i + 1])
        s_svdd[i], s_sad[i], n_lab[i] = f_svdd(z)[0], f_sad(z)[0], len(A)
    return to_rows(lab, pairs, s_svdd), to_rows(lab, pairs, s_sad), n_lab

if HAS_TORCH:
    torch.set_num_threads(4)
    for p, lab in DATA.items():
        for seed in [0, 1, 2]:                          # 신경망은 시작값에 따라 흔들리므로 3번
            a, b, n_lab = sad_scores(lab, BASE[p], seed=seed)
            ROWS.append(report(lab, a, f"Deep SVDD (seed {seed})", p))
            ROWS.append(report(lab, b, f"Deep SAD (seed {seed})", p))
        X, T, pairs, bad, block = shot_view(lab)
        print(p, "블록별 '다른 날 불량' 수:", {int(k): int(v) for k, v in pd.Series(n_lab).groupby(block).max().items()},
              " 블록별 실제 불량 쇼트:", {int(k): int(v) for k, v in pd.Series(bad > 0).groupby(block).sum().items()})
pd.DataFrame(ROWS).round(3)

## 파트 C. 경계선을 확률로 정하기 — 컨포멀

지금까지 경계선은 '점수 상위 5%'였습니다. 이건 **이미 다 본 점수들 안에서** 자른 것이라 현장에서는 미리 알 수 없습니다.
컨포멀은 **과거 쇼트들의 점수 분포**를 기준으로 "이 쇼트보다 점수가 높았던 과거 쇼트의 비율(p값)"을 계산하고, p < 0.05면 경보를 울립니다.
과거와 지금이 같은 분포라면 **양품의 약 5%만** 경보가 울린다는 것이 이론적으로 보장됩니다.

- **일반 컨포멀**: 과거 300쇼트를 똑같이 취급
- **가중 컨포멀**: 최근 쇼트일수록 무게를 크게 (Barber 외 2023의 아이디어, 반감기 30쇼트)

> **질문 9-3.** 공정 상태가 계속 바뀌는(드리프트) 상황에서 일반 컨포멀의 실제 양품 경보율은 5%보다 클까요, 작을까요? 가중 컨포멀은?
>
> 내 예상:

In [ ]:
def stream_scores(base, K=10):
    """비라벨 정상 모드 전체 흐름에 대해, 각 쇼트의 과거기준선(강건 z, K=10) 점수."""
    BX = base[VARS].values
    s = np.full(len(BX), np.nan)
    for i in range(K, len(BX)):
        P = BX[i - K:i]
        med = np.median(P, axis=0)
        sd = np.maximum(np.median(np.abs(P - med), axis=0) * 1.4826, P.std(axis=0)); ok = sd > 1e-6
        z = np.where(ok, np.minimum(np.abs(BX[i] - med) / np.where(ok, sd, 1), 20), 0)
        s[i] = np.sqrt((z ** 2).mean())
    return pd.Series(s, index=base["ts"].values)

rows_c = []
for p, lab in DATA.items():
    st = stream_scores(BASE[p]).dropna()          # 라벨 쇼트도 값으로는 흐름 안에 들어 있음
    X, T, pairs, bad, block = shot_view(lab)
    p_plain, p_w = np.zeros(len(T)), np.zeros(len(T))
    for i, t in enumerate(T):
        cal = st[st.index < t].values[-300:]       # 과거 300쇼트의 점수
        s_now = st.get(t, np.nan)
        if np.isnan(s_now) or len(cal) < 50:
            p_plain[i] = p_w[i] = 1.0; continue
        w = 0.5 ** ((len(cal) - 1 - np.arange(len(cal))) / 30)   # 최근일수록 무게 큼 (반감기 30)
        p_plain[i] = (1 + (cal >= s_now).sum()) / (len(cal) + 1)
        p_w[i] = (1 + (w * (cal >= s_now)).sum()) / (1 + w.sum())
    good = bad == 0
    for name, pv in [("일반 컨포멀", p_plain), ("가중 컨포멀", p_w)]:
        rows_c.append({"제품": p, "방법": name,
                       "양품 쇼트 경보율 (목표 5%)": (pv[good] < 0.05).mean(),
                       "불량 쇼트 경보율": (pv[~good] < 0.05).mean() if (~good).any() else np.nan,
                       "둘다불량 쇼트 경보": f"{int((pv[bad == 2] < 0.05).sum())}/{int((bad == 2).sum())}"})
pd.DataFrame(rows_c).round(3)

### 9단계 정리 (직접 적기)
1. 자기지도(변수 가리고 맞히기)는 강건 z보다 나았나? 무엇을 더 잡았나?
2. 준지도(Deep SAD)는 라벨을 썼는데도 왜 나아지지 않았나? (블록별 '다른 날 불량' 수와 연결)
3. 컨포멀 결과는 현장 운영에 어떤 지침을 주나?
4. 지금까지 모든 방법(지도 2, 비지도 5, 물리 1, 자기지도 1, 준지도 1)의 결론을 한 문장으로.